In [3]:
import os

# Set your token in the environment before running downloads (do not commit secrets).
# Example: export HF_TOKEN=...   (Linux/macOS)   or   set HF_TOKEN=...   (Windows CMD)
HF_TOKEN = ""
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the environment to download from Hugging Face.")

In [8]:
from huggingface_hub import hf_hub_download
import os
import shutil
import tempfile

# <repo>/clartts — under the repository root; only .parquet shards here (HF uses a temp cache).
_cwd = os.getcwd()
if os.path.basename(_cwd.rstrip("/\\")) == "tacotron2":
    _repo = os.path.dirname(_cwd)
else:
    _repo = _cwd
clartts_dir = os.path.normpath(os.path.join(_repo, "clartts"))
os.makedirs(clartts_dir, exist_ok=True)

clartts_files = ["data/test-00000-of-00001.parquet"] + [
    f"data/train-{i:05d}-of-00026.parquet" for i in range(26)
]

downloaded_files = []
_tmp = tempfile.mkdtemp(prefix="hf_clartts_")
try:
    for fname in clartts_files:
        print(f"Downloading {fname} ...")
        file_path = hf_hub_download(
            repo_id="MBZUAI/ClArTTS",
            filename=fname,
            cache_dir=_tmp,
            token=HF_TOKEN,
            repo_type="dataset",
        )
        dest_file = os.path.join(clartts_dir, os.path.basename(fname))
        shutil.copy2(file_path, dest_file)
        downloaded_files.append(dest_file)
finally:
    shutil.rmtree(_tmp, ignore_errors=True)

print(f"Downloaded {len(downloaded_files)} parquet file(s) to {clartts_dir}")

data/test-00000-of-00001.parquet:   0%|          | 0.00/68.7M [00:00<?, ?B/s]

data/train-00000-of-00026.parquet:   0%|          | 0.00/123M [00:00<?, ?B/s]

data/train-00001-of-00026.parquet:   0%|          | 0.00/121M [00:00<?, ?B/s]

data/train-00002-of-00026.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

data/train-00003-of-00026.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

...

data/train-00022-of-00026.parquet:   0%|          | 0.00/121M [00:00<?, ?B/s]

data/train-00023-of-00026.parquet:   0%|          | 0.00/124M [00:00<?, ?B/s]

data/train-00024-of-00026.parquet:   0%|          | 0.00/123M [00:00<?, ?B/s]

data/train-00025-of-00026.parquet:   0%|          | 0.00/119M [00:00<?, ?B/s]

Downloaded 27 parquet file(s) to /workspace/tacotron2-wavernn/clartts


## Combine parquet shards into standard names

After downloading shards into `clartts/`, run the **next cell** to merge them in place into **`combined.parquet`** (all train shards), **`clartts_train.parquet`** / **`clartts_val.parquet`** (random split, default 10% val), and **`clartts_test.parquet`** (if `test-*.parquet` exists). Same layout as `commons/hyperparams.py` `DATASET_PATH` and the dataset section below.

In [9]:
import os
import re
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Same clartts folder as the download cell (recomputed so this cell can run standalone)
_cwd = os.getcwd()
_repo = os.path.dirname(_cwd) if os.path.basename(_cwd.rstrip("/\\")) == "tacotron2" else _cwd
clartts_dir = Path(_repo) / "clartts"

VAL_FRACTION = 0.1
SPLIT_SEED = 42


def _sort_shards(paths, prefix: str):
    def key(p: Path):
        m = re.search(rf"{prefix}-(\d+)-of-", p.name)
        return (int(m.group(1)), p.name) if m else (10**9, p.name)

    return sorted(paths, key=key)


in_dir = clartts_dir
if not in_dir.is_dir():
    raise FileNotFoundError(f"Missing directory: {in_dir}")

train_files = _sort_shards(list(in_dir.glob("train-*.parquet")), "train")
test_files = _sort_shards(list(in_dir.glob("test-*.parquet")), "test")

if not train_files:
    raise FileNotFoundError(f"No train-*.parquet under {in_dir}")

print(f"Reading {len(train_files)} train shard(s)…")
full_train = pd.concat([pd.read_parquet(f) for f in train_files], ignore_index=True)
print(f"  Rows: {len(full_train)}")

combined_path = in_dir / "combined.parquet"
full_train.to_parquet(combined_path, index=False)
print(f"Wrote {combined_path}")

tr_df, va_df = train_test_split(
    full_train,
    test_size=VAL_FRACTION,
    random_state=SPLIT_SEED,
)
train_out = in_dir / "clartts_train.parquet"
val_out = in_dir / "clartts_val.parquet"
tr_df.to_parquet(train_out, index=False)
va_df.to_parquet(val_out, index=False)
print(f"Wrote {train_out} ({len(tr_df)} rows), {val_out} ({len(va_df)} rows)")

if test_files:
    print(f"Reading {len(test_files)} test shard(s)…")
    full_test = pd.concat([pd.read_parquet(f) for f in test_files], ignore_index=True)
    test_out = in_dir / "clartts_test.parquet"
    full_test.to_parquet(test_out, index=False)
    print(f"Wrote {test_out} ({len(full_test)} rows)")
else:
    print("No test-*.parquet shards; skipped clartts_test.parquet")

print("Done combining.")

Reading 26 train shard(s)…
  Rows: 9500
Wrote /workspace/tacotron2-wavernn/clartts/combined.parquet
Wrote /workspace/tacotron2-wavernn/clartts/clartts_train.parquet (8550 rows), /workspace/tacotron2-wavernn/clartts/clartts_val.parquet (950 rows)
Reading 1 test shard(s)…
Wrote /workspace/tacotron2-wavernn/clartts/clartts_test.parquet (205 rows)
Done combining.


## 1. Clone and Set Up the Repository

If running in a fresh environment (e.g., Colab), clone the repository and set up the working directory. If running locally, ensure your working directory is the root of the repository.

In [1]:
# If running in Colab or a new environment, uncomment and run:
# !git clone https://github.com/HazardCodeBolt/tacotron2-wavernn.git
# %cd tacotron2-wavernn

import os

_cwd = os.getcwd()
# If the notebook lives under tacotron2/, move to repository root so imports and data paths match.
if os.path.basename(_cwd.rstrip("\\/")) == "tacotron2":
    os.chdir(os.path.dirname(_cwd))
print("Current working directory:", os.getcwd())

Current working directory: /workspace/tacotron2-wavernn


## 2. Install Dependencies

Install all required Python packages. If running locally, ensure your environment matches requirements.txt. If in Colab, use pip to install dependencies.

In [1]:
# If running in Colab or a new environment, uncomment and run:
! pip install -r requirements.txt
# !apt-get install -y libsndfile1

# If running locally, ensure your virtual environment is activated and dependencies are installed.

  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numba-0.65.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.9 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached pooch-1.9.0-py3

KeyboardInterrupt: 

KeyboardInterrupt: 

## 3. Import Required Libraries

Import all necessary libraries, including PyTorch, NumPy, pandas, matplotlib, and custom modules from this repository.

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repository root (parent of tacotron2/ if cwd is tacotron2, else cwd)
_root = os.getcwd()
if os.path.basename(_root.rstrip("\\/")) == "tacotron2":
    PROJECT_ROOT = os.path.dirname(_root)
else:
    PROJECT_ROOT = _root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Parquet data: <repo>/clartts (see .gitignore). Keep only .parquet files there.
CLARTTS_DATA_DIR = os.path.join(PROJECT_ROOT, "clartts")

from commons.hyperparams import Tacotron2Config, DATASET_PATH
from commons.dataset import TTSDataset, TTSCollator, denormalize
from tacotron2.model import Tacotron2
from tacotron2.tokenizer import Tokenizer

## 4. Prepare Dataset

Parquet data lives in **`clartts/`** at the **repository root** (same folder as `commons/`, `tacotron2/`): HF download shards, optional `clartts_train.parquet` / `clartts_val.parquet`, or `DATASET_PATH` from `commons/hyperparams.py`. Inspect a sample transcript and mel-spectrogram.

In [ ]:
# Load dataset (prefer ClArTTS splits from this repo if present)
config = Tacotron2Config()
tokenizer = Tokenizer()
config.num_chars = tokenizer.vocab_size
config.pad_token_id = tokenizer.pad_token_id

_train_parquet = os.path.join(CLARTTS_DATA_DIR, "clartts_train.parquet")
_manifest = (
    _train_parquet
    if os.path.isfile(_train_parquet)
    else os.path.normpath(os.path.join(PROJECT_ROOT, DATASET_PATH))
)

train_dataset = TTSDataset(
    _manifest,
    sample_rate=config.sample_rate,
    n_fft=config.n_fft,
    window_size=config.win_length,
    hop_size=config.hop_length,
    fmin=config.fmin,
    fmax=config.fmax,
    num_mels=config.n_mels,
    min_db=config.min_db,
    max_scaled_abs=config.max_scaled_abs,
)
collate_fn = TTSCollator()

# Show a sample
sample_transcript, sample_mel = train_dataset[0]
print("Manifest:", _manifest)
print("Sample transcript:", sample_transcript)
print("Mel shape:", sample_mel.shape)
plt.figure(figsize=(10, 4))
plt.imshow(sample_mel, aspect="auto", origin="lower")
plt.title("Sample Mel-Spectrogram")
plt.xlabel("Frames")
plt.ylabel("Mel bins")
plt.colorbar()
plt.show()

## 5. Configure training hyperparameters

Defaults live in **`commons/hyperparams.py`**: module-level audio constants (`SAMPLE_RATE`, `N_MELS`, …) and the **`Tacotron2Config`** dataclass (`batch_size`, `learning_rate`, `epochs`, `grad_clip`, `checkpoint_dir`, `num_workers`, `seed`, model sizes, etc.). The `config` object is created in §4; change the dataclass file or assign on `config` in a notebook cell before this section.

In [ ]:
# Uses Tacotron2Config from commons/hyperparams.py (see §5 markdown).
from torch.utils.data import DataLoader

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

print(
    "Tacotron2Config:",
    f"batch_size={config.batch_size}, lr={config.learning_rate}, epochs={config.epochs}, "
    f"grad_clip={config.grad_clip}, num_workers={config.num_workers}, seed={config.seed}",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=config.num_workers,
)

_val_parquet = os.path.join(CLARTTS_DATA_DIR, "clartts_val.parquet")
val_loader = None
if os.path.isfile(_val_parquet):
    val_dataset = TTSDataset(
        _val_parquet,
        sample_rate=config.sample_rate,
        n_fft=config.n_fft,
        window_size=config.win_length,
        hop_size=config.hop_length,
        fmin=config.fmin,
        fmax=config.fmax,
        num_mels=config.n_mels,
        min_db=config.min_db,
        max_scaled_abs=config.max_scaled_abs,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=config.num_workers,
    )
    print("Validation:", _val_parquet, "n=", len(val_dataset))
else:
    print("No clartts_val.parquet — validation skipped until splits exist.")

## 6. Initialize Tacotron2 and WaveRNN Models

Instantiate Tacotron2 and the optimizer. **Resume:** set `RESUME_CHECKPOINT` to a `.pth` from §8 (same architecture/config as the run that produced it), then run §8 — training continues from the next epoch up to `config.epochs`.

In [ ]:
# Initialize Tacotron2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Tacotron2(config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, eps=config.eps)
print("Device:", device)

# --- Optional: resume from a §8 checkpoint (same config/architecture as the saved run) ---
_ckpt_dir_for_resume = os.path.normpath(
    os.path.join(PROJECT_ROOT, config.checkpoint_dir.replace("./", "").strip("/\\"))
)
RESUME_CHECKPOINT = None  # e.g. os.path.join(_ckpt_dir_for_resume, "tacotron2_epoch_0010.pth")
start_epoch = 0
RESUME_TRAIN_HISTORY = None


def _load_training_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


if RESUME_CHECKPOINT:
    if not os.path.isfile(RESUME_CHECKPOINT):
        raise FileNotFoundError(f"RESUME_CHECKPOINT not found: {RESUME_CHECKPOINT}")
    _rk = _load_training_checkpoint(RESUME_CHECKPOINT)
    model.load_state_dict(_rk["model_state_dict"])
    optimizer.load_state_dict(_rk["optimizer_state_dict"])
    start_epoch = int(_rk.get("epoch", 0))
    RESUME_TRAIN_HISTORY = {
        "train_losses": list(_rk.get("train_losses", [])),
        "val_losses": list(_rk.get("val_losses", [])),
        "mel_losses": list(_rk.get("mel_losses", [])),
        "refined_mel_losses": list(_rk.get("refined_mel_losses", [])),
        "stop_losses": list(_rk.get("stop_losses", [])),
        "batch_epoch_boundaries": list(_rk.get("batch_epoch_boundaries", [0])),
    }
    print(
        f"Loaded {RESUME_CHECKPOINT!r} — completed epochs={start_epoch}. "
        f"§8 runs epochs {start_epoch + 1}..{config.epochs}."
    )
    if start_epoch >= config.epochs:
        print(
            f"Note: completed epochs ({start_epoch}) >= config.epochs ({config.epochs}). "
            "Raise config.epochs (e.g. in commons/hyperparams or §4) before §8."
        )

# WaveRNN vocoder training is a separate pipeline in this repo (see wavernn/). Tacotron2
# outputs mel spectrograms; WaveRNN would be trained on audio conditioned on those mels.

Device: cuda


## 7. Define Loss Functions and Optimizers

Set up the loss functions and optimizers for Tacotron2 (and WaveRNN if used).

In [14]:
import torch.nn.functional as F

def tacotron2_loss(mel_outs, mel_postnet_out, stop_tokens, mel_padded, gate_padded):
    mel_loss = F.mse_loss(mel_outs, mel_padded)
    refined_mel_loss = F.mse_loss(mel_postnet_out, mel_padded)
    stop_loss = F.binary_cross_entropy_with_logits(stop_tokens.reshape(-1,1), gate_padded.reshape(-1,1))
    return mel_loss, refined_mel_loss, stop_loss

## 8. Training Loop for Tacotron2

Train Tacotron2 and record losses. After **each epoch**: checkpoint `tacotron2_epoch_XXXX.pth`, and figures under **`training_viz/epoch_XXXX/`** inside the same checkpoint directory — `loss.png` (epoch-averaged train/val + batch mel/post/stop losses with epoch dividers) and `mel_alignment.png` (actual mel, predicted post-net mel, difference, attention from the first val batch, or train if no val).

**Resume:** If §6 set `RESUME_CHECKPOINT`, the loop continues from the next epoch and restores loss history from that file. Checkpoints saved without per-batch loss keys only partially restore the batch loss plot.

In [ ]:
if RESUME_TRAIN_HISTORY:
    train_losses = list(RESUME_TRAIN_HISTORY["train_losses"])
    val_losses = list(RESUME_TRAIN_HISTORY["val_losses"])
    mel_losses = list(RESUME_TRAIN_HISTORY["mel_losses"])
    refined_mel_losses = list(RESUME_TRAIN_HISTORY["refined_mel_losses"])
    stop_losses = list(RESUME_TRAIN_HISTORY["stop_losses"])
    batch_epoch_boundaries = list(RESUME_TRAIN_HISTORY["batch_epoch_boundaries"])
else:
    train_losses = []
    val_losses = []
    mel_losses = []
    refined_mel_losses = []
    stop_losses = []
    batch_epoch_boundaries = [0]

_ckpt_dir = os.path.normpath(
    os.path.join(PROJECT_ROOT, config.checkpoint_dir.replace("./", "").strip("/\\"))
)
os.makedirs(_ckpt_dir, exist_ok=True)

for epoch in range(start_epoch, config.epochs):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        text_padded, input_lengths, mel_padded, gate_padded, encoder_mask, decoder_mask = batch
        text_padded = text_padded.to(device)
        mel_padded = mel_padded.to(device)
        gate_padded = gate_padded.to(device)
        encoder_mask = encoder_mask.to(device)
        decoder_mask = decoder_mask.to(device)

        optimizer.zero_grad(set_to_none=True)
        mel_outs, mel_postnet_out, stop_tokens, _ = model(
            text_padded, input_lengths, mel_padded, encoder_mask, decoder_mask
        )
        mel_loss, refined_mel_loss, stop_loss = tacotron2_loss(
            mel_outs, mel_postnet_out, stop_tokens, mel_padded, gate_padded
        )
        loss = mel_loss + refined_mel_loss + stop_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.grad_clip)
        optimizer.step()

        epoch_loss += loss.item()
        mel_losses.append(mel_loss.item())
        refined_mel_losses.append(refined_mel_loss.item())
        stop_losses.append(stop_loss.item())

    avg_loss = epoch_loss / max(len(train_loader), 1)
    train_losses.append(avg_loss)
    batch_epoch_boundaries.append(len(mel_losses))

    val_meta = {}
    if val_loader is not None:
        model.eval()
        v_tot = v_mel = v_rmel = v_stop = 0.0
        n_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                text_padded, input_lengths, mel_padded, gate_padded, encoder_mask, decoder_mask = batch
                text_padded = text_padded.to(device)
                mel_padded = mel_padded.to(device)
                gate_padded = gate_padded.to(device)
                encoder_mask = encoder_mask.to(device)
                decoder_mask = decoder_mask.to(device)
                mel_outs, mel_postnet_out, stop_tokens, _ = model(
                    text_padded, input_lengths, mel_padded, encoder_mask, decoder_mask
                )
                mel_loss, refined_mel_loss, stop_loss = tacotron2_loss(
                    mel_outs, mel_postnet_out, stop_tokens, mel_padded, gate_padded
                )
                loss = mel_loss + refined_mel_loss + stop_loss
                v_tot += loss.item()
                v_mel += mel_loss.item()
                v_rmel += refined_mel_loss.item()
                v_stop += stop_loss.item()
                n_batches += 1
        n_batches = max(n_batches, 1)
        val_losses.append(v_tot / n_batches)
        val_meta = {
            "val_loss_this_epoch": val_losses[-1],
            "val_mel_loss_mean": v_mel / n_batches,
            "val_refined_mel_loss_mean": v_rmel / n_batches,
            "val_stop_loss_mean": v_stop / n_batches,
        }
        print(
            f"Epoch {epoch + 1}/{config.epochs} — train {avg_loss:.4f} | val {val_losses[-1]:.4f} "
            f"(mel {v_mel/n_batches:.4f}, post {v_rmel/n_batches:.4f}, stop {v_stop/n_batches:.4f})"
        )
    else:
        print(f"Epoch {epoch + 1}/{config.epochs} — train loss: {avg_loss:.4f}")

    epoch_done = epoch + 1

    _viz_epoch_dir = os.path.join(_ckpt_dir, "training_viz", f"epoch_{epoch_done:04d}")
    os.makedirs(_viz_epoch_dir, exist_ok=True)

    fig_l, axes_l = plt.subplots(2, 1, figsize=(9, 7), height_ratios=[1, 1.2])
    ex = np.arange(1, len(train_losses) + 1)
    axes_l[0].plot(ex, train_losses, "b-o", label="train (epoch avg)", markersize=4)
    if val_losses:
        axes_l[0].plot(ex, val_losses, "r-s", label="val (epoch avg)", markersize=4)
    axes_l[0].set_xlabel("Epoch")
    axes_l[0].set_ylabel("Loss")
    axes_l[0].set_title(f"Loss through epoch {epoch_done}")
    axes_l[0].legend()
    axes_l[0].grid(True, alpha=0.3)
    bx = np.arange(1, len(mel_losses) + 1)
    axes_l[1].plot(bx, mel_losses, label="mel", alpha=0.85, linewidth=0.9)
    axes_l[1].plot(bx, refined_mel_losses, label="post-net mel", alpha=0.85, linewidth=0.9)
    axes_l[1].plot(bx, stop_losses, label="stop", alpha=0.85, linewidth=0.9)
    for j in range(1, len(batch_epoch_boundaries)):
        axes_l[1].axvline(batch_epoch_boundaries[j], color="gray", linestyle="--", alpha=0.45)
    axes_l[1].set_xlabel("Batch index (cumulative)")
    axes_l[1].set_ylabel("Loss")
    axes_l[1].set_title("Batch losses (dashed lines = epoch ends)")
    axes_l[1].legend(loc="upper right", fontsize=8)
    axes_l[1].grid(True, alpha=0.3)
    fig_l.tight_layout()
    fig_l.savefig(os.path.join(_viz_epoch_dir, "loss.png"), dpi=150)
    plt.close(fig_l)

    probe = val_loader if val_loader is not None else train_loader
    model.eval()
    with torch.no_grad():
        text_padded, input_lengths, mel_padded, gate_padded, encoder_mask, decoder_mask = next(
            iter(probe)
        )
        text_padded = text_padded.to(device)
        mel_padded = mel_padded.to(device)
        encoder_mask = encoder_mask.to(device)
        decoder_mask = decoder_mask.to(device)
        _, mel_post, _, attn = model(
            text_padded, input_lengths, mel_padded, encoder_mask, decoder_mask
        )
    true_mel = denormalize(mel_padded[0].T.cpu())
    pred_mel = denormalize(mel_post[0].T.cpu())
    attention = attn[0].T.cpu()
    t_use = min(true_mel.shape[1], pred_mel.shape[1])
    true_mel = true_mel[:, :t_use].detach().cpu().numpy()
    pred_mel = pred_mel[:, :t_use].detach().cpu().numpy()
    attention = attention.detach().cpu().numpy()
    diff_np = np.asarray(pred_mel - true_mel, dtype=np.float64)
    vlim = float(np.abs(diff_np).max()) if diff_np.size else 1.0
    if vlim == 0:
        vlim = 1.0

    fig_m, axes_m = plt.subplots(4, 1, figsize=(9, 14))
    im0 = axes_m[0].imshow(true_mel, aspect="auto", origin="lower", interpolation="none")
    axes_m[0].set_title(f"Actual mel (item 0, {'val' if val_loader else 'train'} probe)")
    axes_m[0].set_ylabel("Mel bin")
    fig_m.colorbar(im0, ax=axes_m[0], fraction=0.02, pad=0.02)
    im1 = axes_m[1].imshow(pred_mel, aspect="auto", origin="lower", interpolation="none")
    axes_m[1].set_title("Predicted mel (post-net)")
    axes_m[1].set_ylabel("Mel bin")
    fig_m.colorbar(im1, ax=axes_m[1], fraction=0.02, pad=0.02)
    im2 = axes_m[2].imshow(
        diff_np,
        aspect="auto",
        origin="lower",
        interpolation="none",
        cmap="RdBu_r",
        vmin=-vlim,
        vmax=vlim,
    )
    axes_m[2].set_title("Difference (pred − actual, dB)")
    axes_m[2].set_ylabel("Mel bin")
    fig_m.colorbar(im2, ax=axes_m[2], fraction=0.02, pad=0.02)
    im3 = axes_m[3].imshow(attention, aspect="auto", origin="lower", interpolation="none")
    axes_m[3].set_title("Alignment (attention)")
    axes_m[3].set_ylabel("Encoder")
    axes_m[3].set_xlabel("Decoder time")
    fig_m.colorbar(im3, ax=axes_m[3], fraction=0.02, pad=0.02)
    fig_m.suptitle(f"Epoch {epoch_done}", y=1.002)
    fig_m.tight_layout()
    fig_m.savefig(
        os.path.join(_viz_epoch_dir, "mel_alignment.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig_m)

    print(f"  Saved plots to {_viz_epoch_dir}")

    ckpt = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "epoch": epoch_done,
        "epochs_total": config.epochs,
        "train_losses": list(train_losses),
        "val_losses": list(val_losses),
        "mel_losses": list(mel_losses),
        "refined_mel_losses": list(refined_mel_losses),
        "stop_losses": list(stop_losses),
        "batch_epoch_boundaries": list(batch_epoch_boundaries),
        "train_loss_this_epoch": avg_loss,
        "manifest_path": _manifest,
        "viz_dir": _viz_epoch_dir,
        "device": str(device),
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    ckpt.update(val_meta)
    _epoch_path = os.path.join(_ckpt_dir, f"tacotron2_epoch_{epoch_done:04d}.pth")
    torch.save(ckpt, _epoch_path)
    print(f"  Checkpoint saved: {_epoch_path}")

## 9. WaveRNN (vocoder)

WaveRNN is not trained in this notebook. After Tacotron2 produces mel spectrograms, train the vocoder with the `wavernn/` scripts. Tacotron2 teacher-forcing uses ground-truth mels during training.

## 10. Save latest alias (optional)

Per-epoch checkpoints are already in §8. This copies the same payload as the last epoch to `tacotron2_notebook_last.pth` for a stable filename.

In [ ]:
_last = os.path.join(_ckpt_dir, f"tacotron2_epoch_{config.epochs:04d}.pth")
ckpt_path = os.path.join(_ckpt_dir, "tacotron2_notebook_last.pth")
if os.path.isfile(_last):
    import shutil
    shutil.copy2(_last, ckpt_path)
    print("Latest epoch checkpoint copied to", ckpt_path)
else:
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "epoch": config.epochs,
            "epochs_total": config.epochs,
            "train_losses": list(train_losses),
            "val_losses": list(val_losses),
            "manifest_path": _manifest,
            "device": str(device),
            "learning_rate": optimizer.param_groups[0]["lr"],
        },
        ckpt_path,
    )
    print("Saved fallback checkpoint to", ckpt_path, "(run §8 first for per-epoch files)")

## 11. Visualize Training Progress

Plot the loss curves to monitor training progress.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train (epoch avg)")
if val_losses:
    plt.plot(val_losses, label="Val (epoch avg)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Tacotron2 training / validation loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(mel_losses, label='Mel Loss')
plt.plot(refined_mel_losses, label='Refined Mel Loss')
plt.plot(stop_losses, label='Stop Loss')
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('Tacotron2 Batch Losses')
plt.legend()
plt.show()